# ETL documentaire PySpark - Assistant multi-agents assurance-vie

Ce notebook simule le traitement d'une procédure interne de rachat total : extraction, nettoyage, exclusion de contenus sensibles ou obsolètes, enrichissement, découpage et préparation à l'indexation RAG. Les données sont entièrement fictives.

In [ ]:
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.window import Window

spark = SparkSession.builder.appName('ETL_Procedure_Assurance_Vie').getOrCreate()
spark.sparkContext.setLogLevel('WARN')

## 1. Extraction - document brut fictif
Dans un environnement réel, cette étape lirait un PDF/Word depuis une GED. Ici, chaque section extraite est représentée par une ligne.

In [ ]:
raw_sections = [
    ('AV-RACHAT-014', 'Objet', 'Cette procédure décrit les étapes d un rachat total d un contrat assurance-vie.', '2026-05-15', '3.2', 'valide'),
    ('AV-RACHAT-014', 'Conditions', 'Le contrat doit être actif. Le client doit être identifié et la demande signée. Aucun blocage judiciaire ou réglementaire ne doit être présent.', '2026-05-15', '3.2', 'valide'),
    ('AV-RACHAT-014', 'Pièces justificatives - client majeur', 'Demander le formulaire signé, une pièce d identité valide et un RIB au nom du souscripteur.', '2026-05-15', '3.2', 'valide'),
    ('AV-RACHAT-014', 'Pièces justificatives - mineur', 'Demander le formulaire signé par les représentants légaux, les pièces d identité, le justificatif de représentation et le RIB.', '2026-05-15', '3.2', 'valide'),
    ('AV-RACHAT-014', 'Délais', 'Après réception d un dossier complet, le délai indicatif est de 10 à 15 jours ouvrés. Ne jamais garantir un délai ferme.', '2026-05-15', '3.2', 'valide'),
    ('AV-RACHAT-014', 'Dossier incomplet', 'Lister précisément les pièces manquantes. Le délai est suspendu jusqu à réception et validation des pièces.', '2026-05-15', '3.2', 'valide'),
    ('AV-RACHAT-014', 'Décès du souscripteur', 'Ne pas traiter comme un rachat. Orienter vers la procédure Décès et règlement du contrat.', '2026-05-15', '3.2', 'valide'),
    ('AV-RACHAT-014', 'Réponse autorisée', 'Nous accusons réception de votre demande. Une fois le dossier complet et validé, le délai indicatif est de 10 à 15 jours ouvrés.', '2026-05-15', '3.2', 'valide'),
    ('AV-RACHAT-014', 'Informations internes confidentielles', 'Contact interne : support-assurancevie-interne@exemple.fr. Seuil anti-fraude interne : confidentiel.', '2026-05-15', '3.2', 'valide'),
    ('AV-RACHAT-014', 'Ancienne note', 'Version 1.4 - 2019 : le délai maximum de traitement est de 30 jours. Cette note est obsolète.', '2019-01-01', '1.4', 'obsolète')
]

schema = T.StructType([
    T.StructField('document_id', T.StringType(), False),
    T.StructField('section_title', T.StringType(), False),
    T.StructField('raw_text', T.StringType(), False),
    T.StructField('document_date', T.StringType(), False),
    T.StructField('version', T.StringType(), False),
    T.StructField('status', T.StringType(), False),
])

raw_df = spark.createDataFrame(raw_sections, schema)
raw_df.show(truncate=False)

## 2. Transformation - nettoyage, règles de sécurité et exclusion des contenus obsolètes
Les sections confidentielles et les anciennes versions ne doivent pas alimenter l'index RAG. Les données personnelles sont masquées avant tout chargement.

In [ ]:
sensitive_pattern = r'(?i)(confidentiel|anti-fraude|contact interne|@exemple\\.fr|iban|numéro de contrat)'

clean_df = (raw_df
    .withColumn('document_date', F.to_date('document_date'))
    .withColumn('clean_text', F.trim(F.regexp_replace('raw_text', r'\\s+', ' ')))
    .withColumn('contains_sensitive_data', F.col('clean_text').rlike(sensitive_pattern))
    .withColumn('is_current_version', F.col('status') == 'valide')
    .withColumn('is_indexable', F.col('is_current_version') & ~F.col('contains_sensitive_data'))
)

clean_df.select('section_title', 'status', 'contains_sensitive_data', 'is_indexable').show(truncate=False)

## 3. Enrichissement - métadonnées métier
Les métadonnées permettent de filtrer la recherche par produit, thème, version, statut et niveau de confidentialité.

In [ ]:
enriched_df = (clean_df
    .filter('is_indexable')
    .withColumn('product', F.lit('assurance-vie'))
    .withColumn('theme', F.lit('rachat-total'))
    .withColumn('confidentiality', F.lit('interne'))
    .withColumn('source_name', F.lit('Procedure interne rachat total'))
    .withColumn('source_reference', F.concat(F.lit('procedure://'), F.col('document_id'), F.lit('/'), F.regexp_replace(F.lower('section_title'), ' ', '-')))
)

enriched_df.select('section_title', 'product', 'theme', 'version', 'document_date', 'confidentiality').show(truncate=False)

## 4. Découpage en chunks
Pour cette démonstration, une section correspond à un chunk. En production, une section longue serait découpée en fragments de taille fixe avec recouvrement, tout en conservant le contexte et la référence source.

In [ ]:
chunks_df = (enriched_df
    .withColumn('chunk_id', F.concat_ws('_', F.col('document_id'), F.monotonically_increasing_id()))
    .withColumn('chunk_text', F.col('clean_text'))
    .select('chunk_id', 'document_id', 'section_title', 'chunk_text', 'product', 'theme', 'version', 'document_date', 'status', 'confidentiality', 'source_name', 'source_reference')
)

chunks_df.show(truncate=False)

## 5. Contrôles qualité avant chargement
Un pipeline doit empêcher l'indexation d'un chunk sans texte, sans métadonnées essentielles, ou issu d'une version non valide.

In [ ]:
quality_checks = chunks_df.select(
    F.count('*').alias('nb_chunks'),
    F.sum(F.when(F.length('chunk_text') == 0, 1).otherwise(0)).alias('chunks_vides'),
    F.sum(F.when(F.col('source_reference').isNull(), 1).otherwise(0)).alias('sources_manquantes'),
    F.sum(F.when(F.col('status') != 'valide', 1).otherwise(0)).alias('versions_non_valides')
)
quality_checks.show()

assert quality_checks.first()['chunks_vides'] == 0
assert quality_checks.first()['sources_manquantes'] == 0
assert quality_checks.first()['versions_non_valides'] == 0
print('Contrôles qualité réussis : jeu de données prêt à indexer.')

## 6. Chargement - export Delta/Parquet prêt pour l'index RAG
Dans un projet Databricks, la table Delta peut être consommée par une étape de génération d'embeddings puis chargée dans Azure AI Search.

In [ ]:
output_path = './data/gold/rag_document_chunks'
chunks_df.write.mode('overwrite').format('parquet').save(output_path)

# Exemple Databricks : remplacer parquet par delta si Delta Lake est disponible
# chunks_df.write.mode('overwrite').format('delta').saveAsTable('gold.rag_document_chunks')

print(f'Chargement terminé : {output_path}')
display(chunks_df)  # Dans Databricks ; remplacer par chunks_df.show() en local